#### **bioconductor**

In [1]:
import json, csv, re, hashlib
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple, Union
import pandas as pd

In [2]:
def _norm_ws(s: Optional[str]) -> str:
    if not s:
        return ""
    import re as _re
    return _re.sub(r"\s+", " ", s).strip()

def _list_to_semicolon(items: Any) -> str:
    if not items:
        return ""
    if isinstance(items, list):
        return ";".join(str(x) for x in items if str(x).strip())
    return str(items)

def _sha1(s: str) -> str:
    import hashlib as _hashlib
    return _hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def _dedup_keep_order(seq: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in seq:
        if x in seen: 
            continue
        seen.add(x)
        out.append(x)
    return out

In [3]:
# ----------------------------
# JSON/JSONL loader
# ----------------------------
def _load_json_any(path: Union[str, Path]) -> List[Dict[str, Any]]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Input not found: {p}")
    txt = p.read_text(encoding="utf-8", errors="ignore")

    lines = [ln for ln in txt.splitlines() if ln.strip()]
    is_jsonl = False
    try:
        first_trim = lines[0].lstrip()
        is_jsonl = first_trim.startswith("{") and not txt.lstrip().startswith("[")
    except Exception:
        pass

    if is_jsonl:
        out = []
        for ln in lines:
            try:
                out.append(json.loads(ln))
            except Exception:
                continue
        return out
    obj = json.loads(txt)
    if isinstance(obj, list):
        return obj
    return [obj]

# ----------------------------
# Candidate selection logic
# ----------------------------
def _select_candidate(answers_all: List[Dict[str, Any]], *, min_score: int = 2) -> Tuple[Optional[Dict[str, Any]], str]:
    if not answers_all:
        return None, "no_answers"
    accepted = [a for a in answers_all if a.get("accepted") and _norm_ws(a.get("text"))]
    if accepted:
        accepted_sorted = sorted(
            accepted,
            key=lambda a: (int(a.get("score", 0) or 0), len(_norm_ws(a.get("text") or ""))),
            reverse=True
        )
        return accepted_sorted[0], "accepted"
    scored = [a for a in answers_all if int(a.get("score", 0) or 0) >= min_score and _norm_ws(a.get("text"))]
    if scored:
        scored_sorted = sorted(
            scored,
            key=lambda a: (int(a.get("score", 0) or 0), len(_norm_ws(a.get("text") or ""))),
            reverse=True
        )
        return scored_sorted[0], f"score>={min_score}"
    return None, "no_candidate_meets_rules"

In [4]:
# ----------------------------
# Columns (now using 'question','answer' + 'tagn')
# ----------------------------
_DEFAULT_COLUMNS = [
    "post_id","url","title","question","answer",
    "accepted","score","n_answers","tags","tagn","topic_labels",
    "created_at","view_count","selection_reason"
]

def _merge_tags(rec: Dict[str, Any], crawl_tag_keys: List[str]) -> List[str]:
    tags = rec.get("tags_site") or rec.get("tags_html") or []
    merged = []
    if isinstance(tags, list):
        merged.extend([t for t in tags if t])
    elif isinstance(tags, str):
        merged.extend([t.strip() for t in re.split(r"[,\s;]+", tags) if t.strip()])

    for k in crawl_tag_keys:
        v = rec.get(k)
        if not v:
            continue
        if isinstance(v, list):
            merged.extend([str(x).strip() for x in v if str(x).strip()])
        else:
            merged.append(str(v).strip())

    seen_lower = set()
    deduped = []
    for t in merged:
        tl = t.lower()
        if tl in seen_lower:
            continue
        seen_lower.add(tl)
        deduped.append(t)
    return deduped

def _build_row(rec: Dict[str, Any], candidate: Dict[str, Any], selection_reason: str,
               crawl_tag_keys: List[str]) -> Dict[str, Any]:
    base_tags = rec.get("tags_site") or rec.get("tags_html") or []
    topic_labels = rec.get("topic_labels") or []
    merged_tags = _merge_tags(rec, crawl_tag_keys=crawl_tag_keys)

    row = {
        "post_id": rec.get("post_id"),
        "url": rec.get("url"),
        "title": _norm_ws(rec.get("title") or ""),
        # NOTE: keys renamed to 'question' and 'answer' to match your include_columns
        "question": _norm_ws(rec.get("question") or ""),
        "answer": _norm_ws(candidate.get("text") or ""),
        "accepted": bool(candidate.get("accepted", False)),
        "score": int(candidate.get("score", 0) or 0),
        "n_answers": len(rec.get("answers_all") or []),
        "tags": _list_to_semicolon(base_tags),
        "tagn": _list_to_semicolon(merged_tags),
        "topic_labels": _list_to_semicolon(topic_labels),
        "created_at": rec.get("created_at") or rec.get("question_created_at"),
        "view_count": int(rec.get("view_count", 0) or 0),
        "selection_reason": selection_reason,
    }
    return row

def process_bioconductor_json(
    json_path: Union[str, Path],
    out_csv_path: Union[str, Path],
    *,
    min_q_len: int = 30,
    min_a_len: int = 40,
    min_score: int = 2,
    include_columns: Optional[List[str]] = None,
    dedup_by_question: bool = True,
    crawl_tag_keys: Optional[List[str]] = None,
    required_any_tags: Optional[List[str]] = None
) -> Path:
    if crawl_tag_keys is None:
        crawl_tag_keys = ["search_query", "crawl_tag"]

    records = _load_json_any(json_path)
    rows: List[Dict[str, Any]] = []
    seen_q_hash = set()

    required_norm = [t.strip().lower() for t in (required_any_tags or []) if str(t).strip()] or None

    for rec in records:
        q = _norm_ws(rec.get("question") or "")
        if len(q) < min_q_len:
            continue
        answers_all = rec.get("answers_all") or []
        candidate, reason = _select_candidate(answers_all, min_score=min_score)
        if not candidate:
            continue
        if len(_norm_ws(candidate.get("text") or "")) < min_a_len:
            continue

        row = _build_row(rec, candidate, reason, crawl_tag_keys=crawl_tag_keys)

        if required_norm:
            tokens = [tok.strip().lower() for tok in row["tagn"].split(";") if tok.strip()]
            if not any(any(req in tok for tok in tokens) for req in required_norm):
                continue

        if dedup_by_question:
            h = _sha1(row["question"])
            if h in seen_q_hash:
                continue
            seen_q_hash.add(h)

        rows.append(row)

    # Column selection
    if include_columns is None:
        include_columns = _DEFAULT_COLUMNS
    else:
        include_columns = _dedup_keep_order([c for c in include_columns if c in _DEFAULT_COLUMNS])

    out_csv_path = Path(out_csv_path)
    out_csv_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_csv_path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=include_columns)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in include_columns})

    return out_csv_path

def preview_csv(csv_path: Union[str, Path], name: str = "bioconductor QA Preview"):
    p = Path(csv_path)
    if not p.exists():
        raise FileNotFoundError(f"CSV not found: {p}")
    df = pd.read_csv(p)
    return df

In [5]:
# RUN

In [34]:
process_bioconductor_json(
    "/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/raw/bioconductor/bioconductor_metagenomics_tag.jsonl",
    "/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/bioconductor/bioconductor_metagenomics_tag.csv",
    include_columns=["question","answer","tagn","accepted","score","url"],
    min_score=1,
    crawl_tag_keys=["search_query"], 
    required_any_tags=None
)

PosixPath('/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/bioconductor/bioconductor_metagenomics_tag.csv')

In [35]:
df = preview_csv("/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/bioconductor/bioconductor_metagenomics_tag.csv")
df.head()

,question,answer,tagn,accepted,score,url
0,"Hi everybody, I am fairly new to NGS analysis,...",Thanks for sending your sessionInfo(). I took ...,"metagenomics,chipseq,biomart,shortread,convert...",False,1,https://support.bioconductor.org/p/61517/
1,"Hello, I am trying to analyze a dataset of bac...","I have an error message saying ""Error in .next...","bayseq,metagenomics;metagenomics",True,1,https://support.bioconductor.org/p/62821/
2,"Hi everyone, I have data from metagenomic expe...",Question 1: I don't think there's enough non-z...,"edger,metagenomics;metagenomics",False,1,https://support.bioconductor.org/p/68940/
3,"Dear Bioconductor community, I'm currently exp...","Briefly, filtering reduces the severity of the...","edger,metagenomics,gene;abundance,gene;filteri...",True,3,https://support.bioconductor.org/p/73298/
4,"Hello, I have recently been trying to analyse ...",DESeq2 is a tool to test statistical hypothese...,"software;error,deseq2,metagenomics;metagenomics",True,1,https://support.bioconductor.org/p/89746/


In [28]:
df.shape

(1090, 6)

In [29]:
df[df['accepted'] == True].shape

(481, 6)

In [33]:
df_ = df[df['accepted'] == True]
df_.to_csv("/home/biolab-office-1/DATALAB/2025/Genomeer/dataset/04-output/hq/bioconductor/bioconductor_db.csv")
df_.head()

,question,answer,tagn,accepted,score,url
1,"Hello, I am trying to analyze a dataset of bac...","I have an error message saying ""Error in .next...","bayseq,metagenomics;metagenomics",True,1,https://support.bioconductor.org/p/62821/
3,"Dear Bioconductor community, I'm currently exp...","Briefly, filtering reduces the severity of the...","edger,metagenomics,gene;abundance,gene;filteri...",True,3,https://support.bioconductor.org/p/73298/
4,"Hello, I have recently been trying to analyse ...",DESeq2 is a tool to test statistical hypothese...,"software;error,deseq2,metagenomics;metagenomics",True,1,https://support.bioconductor.org/p/89746/
5,I am having metagenomic data from soil samples...,Can you describe the distribution of the count...,"deseq2,metagenomics;metagenomics",True,1,https://support.bioconductor.org/p/89982/
8,Hi I am currently working on some metagenome d...,"hi Mehdi, The way to normalize in DESeq2 if yo...","deseq2,metagenomics;metagenomics",True,2,https://support.bioconductor.org/p/99918/
